# Практическая работа №3
**Тема:** Очистка и исследование данных о продажах видеоигр
**Выполнил:** Шумилин Демьян

## Задание 1. Загрузка данных

In [1]:
import pandas as pd
import matplotlib.pyplot as plt

# Загружаем датасет
df = pd.read_csv('vgsales.csv')
df.head(10)

**Ответ к Заданию 1:** Данные успешно загружены в DataFrame `df`.

## Задание 2. Исследование структуры данных

In [2]:
print("Количество строк и столбцов:", df.shape)
print("\nСписок столбцов:", list(df.columns))
print("\nТипы данных:")
print(df.dtypes)
print("\nОсновные статистические характеристики:")
df.describe()

**Ответы к Заданию 2:**
* **Числовые столбцы:** `Rank`, `Year`, `NA_Sales`, `EU_Sales`, `JP_Sales`, `Other_Sales`, `Global_Sales`
* **Текстовые столбцы:** `Name`, `Platform`, `Genre`, `Publisher`
* **Данные о продажах:** `NA_Sales`, `EU_Sales`, `JP_Sales`, `Other_Sales`, `Global_Sales`

## Задание 3. Поиск пропущенных значений

In [3]:
# Пропущенные значения по столбцам
null_counts = df.isnull().sum()
print("Пропуски по столбцам:\n", null_counts)
print(f"\nБольше всего пропусков в столбце: {null_counts.idxmax()} ({null_counts.max()})")

# Строки с пропущенным годом
df[df['Year'].isnull()].head()

**Ответ к Заданию 3:** Пропуски присутствуют в столбцах `Year` (271) и `Publisher` (58). Наибольшее количество пропусков — в столбце `Year`.

## Задание 4. Очистка данных

In [4]:
# Удаление строк с пропущенным Year
df_clean = df.dropna(subset=['Year']).copy()

print(f"Количество строк до очистки: {len(df)}")
print(f"Количество строк после очистки: {len(df_clean)}")
print(f"Удалено строк: {len(df) - len(df_clean)}")

**Ответы к Заданию 4:**
1. В исходном датасете было 16 598 строк.
2. В очищенном датасете осталось 16 327 строк (удалена 271 строка с пропущенным годом).
3. Пропущенные значения мешают проведению аналитики по временным периодам (трендам по годам) и строительству корректных временных рядов.

## Задание 5. Изучение категориальных признаков

In [5]:
print("Уникальных жанров:", df_clean['Genre'].nunique())
print("Уникальных платформ:", df_clean['Platform'].nunique())
print("Уникальных издателей:", df_clean['Publisher'].nunique())

**Ответ к Заданию 5:** Наибольшее количество уникальных значений имеет признак `Publisher` (576 издателей).

## Задание 6 и 7. Анализ и продажи по жанрам

In [6]:
# Игры по жанрам
genre_counts = df_clean['Genre'].value_counts()
print("Количество игр по жанрам:\n", genre_counts)

# Продажи по жанрам
genre_sales = df_clean.groupby('Genre')['Global_Sales'].sum().sort_values(ascending=False)
print("\nСуммарные мировые продажи по жанрам (млн):\n", genre_sales)

**Ответ к Заданиям 6-7:** Самым популярным жанром по количеству игр и по суммарным продажам является **Action**.

## Задание 8 и 9. Самые продаваемые игры и анализ платформ

In [7]:
# Топ-10 продаваемых игр
top_games = df_clean.sort_values(by='Global_Sales', ascending=False)[['Name', 'Platform', 'Genre', 'Global_Sales']].head(10)
print("--- Топ-10 самых продаваемых игр ---")
print(top_games)

# Платформы по числу игр и продажам
platform_counts = df_clean['Platform'].value_counts().head(10)
platform_sales = df_clean.groupby('Platform')['Global_Sales'].sum().sort_values(ascending=False).head(10)
print("\n--- Топ-10 платформ по числу игр ---\n", platform_counts)
print("\n--- Топ-10 платформ по продажам ---\n", platform_sales)

**Ответ к Заданиям 8-9:** Самая продаваемая игра — **Wii Sports**. По общему объёму продаж лидирует платформа **PS2**.

## Задание 10. Визуализация продаж по жанрам

In [8]:
plt.figure(figsize=(10, 5))
genre_sales.sort_values().plot(kind='barh', color='skyblue', edgecolor='black')
plt.title('Мировые продажи по жанрам (млн копий)')
plt.xlabel('Продажи (млн)')
plt.ylabel('Жанр')
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.show()

**Ответы к Заданию 10:**
1. Наибольшие продажи у жанров **Action** и **Sports**.
2. Наименьшие продажи у жанров **Strategy** и **Puzzle**.
3. Продажи экшен-игр почти в 10 раз превышают продажи стратегий.

## Задание 11. Корреляция продаж

In [9]:
corr_value = df_clean['NA_Sales'].corr(df_clean['Global_Sales'])
print(f"Коэффициент корреляции между NA_Sales и Global_Sales: {corr_value:.4f}")

**Ответы к Заданию 11:**
* Коэффициент корреляции **положительный** (приблизительно `0.941`).
* Значение близко к 1, что указывает на крайне сильную линейную взаимосвязь между продажами в Северной Америке и глобальными продажами.

## Задание 12. Диаграмма рассеяния

In [10]:
plt.figure(figsize=(8, 5))
plt.scatter(df_clean['NA_Sales'], df_clean['Global_Sales'], alpha=0.5, color='blue')
plt.title('Диаграмма рассеяния: NA_Sales vs Global_Sales')
plt.xlabel('Продажи в Северной Америке (млн)')
plt.ylabel('Мировые продажи (млн)')
plt.grid(True)
plt.show()

**Ответы к Заданию 12:**
1. Каждая точка обозначает отдельную видеоигру.
2. Наблюдается четкий тренд: чем выше продажи в Северной Америке, тем выше мировые продажи.
3. Диаграмма полностью подтверждает высокий коэффициент корреляции.

## Задание 13. Корреляция между регионами

In [11]:
sales_cols = ['NA_Sales', 'EU_Sales', 'JP_Sales', 'Other_Sales', 'Global_Sales']
df_clean[sales_cols].corr()

**Ответ к Заданию 13:** Самая сильная корреляция наблюдается между `NA_Sales` и `EU_Sales` (~0.768). Наименьшая корреляция с остальными регионами у `JP_Sales` (Япония), что отражает уникальность и специфику японского игрового рынка.